# Metamorphic sweep with a local model (Qwen2.5-3B via Ollama)

Runs the full metamorphic matrix with the agent graph's judgment steps routed
through a local model on the Kaggle GPU.

**Before running**
1. Settings → Accelerator → **GPU T4 x2** (or P100)
2. Settings → **Internet: On** (needed for `ollama pull`)
3. Attach the project's **private Kaggle Dataset** via *Add Input* (the notebook finds it automatically -- no path to set)
4. Run via **Save & Run All (Commit)** — an interactive session will time out

The notebook smoke-tests first and stops early if the model's output is not
usable, so a bad prompt costs minutes rather than the whole session.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!free -g | sed -n '2p'
!df -h /kaggle/working | tail -1

## 1. Install Ollama and start the server

Ollama 0.40+ releases as `.tar.zst`, and its installer aborts on images without the `zstd` binary (Kaggle's is one). This cell installs `zstd` first and falls back to extracting the release with the `zstandard` wheel if apt is unavailable.


In [ ]:
import os, shutil, subprocess, sys, time, urllib.request

# Ollama 0.40+ ships its release as .tar.zst, and its install script aborts
# with "This version requires zstd for extraction" on images that lack the
# zstd binary -- which Kaggle's does. Install zstd first, then fall back to
# extracting the tarball ourselves with the zstandard wheel if apt is
# unavailable in this image.

def have(binary):
    return shutil.which(binary) is not None

if not have("zstd"):
    print("installing zstd ...")
    subprocess.run("apt-get -qq update && apt-get -qq install -y zstd",
                   shell=True, check=False)
print("zstd:", shutil.which("zstd") or "NOT AVAILABLE")

if not have("ollama"):
    if have("zstd"):
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh",
                       shell=True, check=False)
    if not have("ollama"):
        # Fallback: no zstd binary, so decompress with the Python wheel and
        # unpack into /usr/local ourselves. Same layout the script produces.
        print("falling back to manual extraction ...")
        subprocess.run(f"{sys.executable} -m pip -q install zstandard",
                       shell=True, check=False)
        import tarfile, io, zstandard
        url = ("https://github.com/ollama/ollama/releases/latest/download/"
               "ollama-linux-amd64.tar.zst")
        print("downloading", url)
        raw = urllib.request.urlopen(url, timeout=1800).read()
        decompressed = zstandard.ZstdDecompressor().stream_reader(io.BytesIO(raw))
        with tarfile.open(fileobj=decompressed, mode="r|") as tar:
            tar.extractall("/usr/local")
        os.environ["PATH"] = "/usr/local/bin:" + os.environ.get("PATH", "")

assert have("ollama"), (
    "ollama is still not on PATH. Check the output above: the usual causes "
    "are Internet being off in Settings, or the release layout changing."
)
print("ollama:", shutil.which("ollama"))
subprocess.run(["ollama", "--version"], check=False)

# The server must run in the background for the whole session.
server = subprocess.Popen(["ollama", "serve"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def wait_for_server(timeout=180):
    for _ in range(timeout):
        try:
            urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
            return True
        except Exception:
            time.sleep(1)
    return False

assert wait_for_server(), "ollama server did not come up"
print("ollama server is up")

## 2. Pull the model and record its identity

The model is the **third reproducibility artifact**, alongside the frozen data
snapshot and the recorded seeds. Its digest is written into the run metadata so
a later run can prove it used the same weights.

In [ ]:
MODEL = "qwen2.5:3b-instruct-q4_K_M"

!ollama pull {MODEL}

import json, urllib.request
info = json.loads(urllib.request.urlopen(
    urllib.request.Request("http://localhost:11434/api/show",
                           data=json.dumps({"model": MODEL}).encode(),
                           headers={"Content-Type": "application/json"})
).read())
print("family :", info.get("details", {}).get("family"))
print("params :", info.get("details", {}).get("parameter_size"))
print("quant  :", info.get("details", {}).get("quantization_level"))
MODEL_DIGEST = info.get("model_info", {}).get("general.basename", "") or MODEL
print("digest :", MODEL_DIGEST)

## 3. Put the project on the path

In [ ]:
import shutil, sys, os
from pathlib import Path

# Locate the project under /kaggle/input instead of hard-coding a path.
# A dataset's folder name comes from its title's slug, and Kaggle sometimes
# flattens an uploaded zip's top level, so the path is not predictable. We
# identify the project by its contents: the loader module plus the frozen
# snapshot the loader checksums.
def find_project(root="/kaggle/input"):
    root = Path(root)
    if not root.exists():
        raise SystemExit(f"{root} does not exist -- attach the dataset first")
    for candidate in [root, *root.rglob("*")]:
        if not candidate.is_dir():
            continue
        if ((candidate / "src" / "loading.py").exists()
                and (candidate / "data" / "raw" / "hyderabad.csv").exists()):
            return candidate
    listing = "\n".join(f"  {q}" for q in sorted(root.rglob("*"))[:40])
    raise SystemExit(
        "could not find the project under /kaggle/input.\n"
        "Looked for a directory containing both src/loading.py and "
        "data/raw/hyderabad.csv.\nWhat is attached:\n" + listing
    )

PROJECT_DIR = find_project()
print("found project at:", PROJECT_DIR)

WORK = Path("/kaggle/working/tse_assignment")
if WORK.exists():
    shutil.rmtree(WORK)
shutil.copytree(PROJECT_DIR, WORK)
os.chdir(WORK)
sys.path.insert(0, str(WORK))

!pip -q install -r requirements.txt

from src.loading import load_canonical, verify_snapshot
print("snapshot sha256:", verify_snapshot()[:16], "...")
frame, _ = load_canonical()
print("canonical listings:", len(frame), "(expected 4578)")
assert len(frame) == 4578, "snapshot did not load as expected"
print("project is ready")

## 4. Smoke test — stop early if the model is not usable

Three base inputs, one seed. The number that matters is the **fallback rate**:
the share of calls where the model never produced parseable, valid output and
the agent had to use a recorded default. A high rate means the LLM path is
weaker than it looks, and the report has to say so.

In [ ]:
from src.agents import AgentMode, GraphConfig, build_graph
from src.llm import LLMClient, OllamaBackend
from src.mr.base_inputs import generate_base_inputs
from src.mr.harness import run_relation
from src.mr.relations import ALL_RELATIONS

llm = LLMClient(backend=OllamaBackend(), model=MODEL,
                cache_path=Path("/kaggle/working/llm_cache.json"))
config = GraphConfig(agent_mode=AgentMode.LLM, llm_model=MODEL)
graph = build_graph(config.n_retrieval_agents)
inputs = generate_base_inputs(frame)

for mr in ALL_RELATIONS:
    r = run_relation(mr, frame, inputs[0], 11, graph=graph, config=config, llm=llm)
    print(f"  {mr.mr_id:<6} violated={str(r.violated):<5} {r.seconds:6.1f}s  {r.detail[:70]}")

stats = llm.stats.as_dict()
print("\nLLM accounting:", stats)
assert stats["llm_fallback_rate"] < 0.5, (
    f"fallback rate {stats['llm_fallback_rate']:.2f} is too high to trust; "
    "fix the prompts before spending the session on a full sweep")
print("smoke test OK")

## 5. Determinism probe

Temperature 0 with a fixed seed *should* be deterministic, but that is an
assumption worth measuring rather than asserting — the recorded seeds only pin
behaviour if it holds. The result goes into the report either way.

In [ ]:
from src.llm import LLMClient, OllamaBackend

probe = LLMClient(backend=OllamaBackend(), model=MODEL, cache_path=None)
PROMPT = ("Reply with ONLY a JSON object: "
          '{"suspect_months": ["2023-01"], "reason": "a short reason"}\n\nJSON:')

replies = []
for _ in range(20):
    probe._cache.clear()              # force a real call each time
    replies.append(probe._raw(PROMPT, seed=11, attempt=0))

unique = set(replies)
print(f"identical replies: {replies.count(replies[0])}/20")
print(f"distinct outputs : {len(unique)}")
if len(unique) > 1:
    print("\nNOT fully deterministic at temperature 0. Examples:")
    for u in list(unique)[:3]:
        print("  --", u[:120].replace("\n", " "))
DETERMINISM = {"trials": 20, "distinct_outputs": len(unique),
               "identical_rate": replies.count(replies[0]) / 20}
print(DETERMINISM)

## 6. Full sweep

11 relations x 36 base inputs x 3 seeds. Expect 1–3 hours; the response cache
makes repeated prompts free, so throughput rises as it warms.

In [ ]:
!python scripts/run_harness.py --tag llm --llm --llm-model {MODEL} \
    --llm-cache /kaggle/working/llm_cache.json

## 7. Collect the outputs

In [ ]:
import json, shutil
from pathlib import Path

out = Path("/kaggle/working/outputs")
out.mkdir(exist_ok=True)
shutil.copytree("results/llm", out / "results_llm", dirs_exist_ok=True)
shutil.copy("/kaggle/working/llm_cache.json", out / "llm_cache.json")
(out / "determinism_probe.json").write_text(json.dumps(DETERMINISM, indent=2))

print(open("results/llm/headline.txt").read())
print("\nfiles to download:")
for p in sorted(out.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(out)}  ({p.stat().st_size/1e3:.0f} KB)")